In [118]:
import json
from pathlib import Path

import pandas as pd
import requests

In [119]:
# project folders — relative to project root
PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "data_raw"
PROCESSED_DIR = PROJECT_DIR / "data_processed"

RAW_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(exist_ok=True)

# Open-Meteo terms: https://open-meteo.com/en/terms
# License: CC BY 4.0 — https://open-meteo.com/
# Historical data derived from ERA5 reanalysis (ECMWF)

print("Project folder:", PROJECT_DIR)

Project folder: /Users/aanchal


In [120]:
# Chicago location and date range
LAT = 41.88
LON = -87.63

START_DATE = "2025-01-01"
END_DATE = "2025-06-30"

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "timezone": "America/Chicago",
    "daily": ",".join([
        "weather_code",
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "precipitation_hours"
    ])
}

params

{'latitude': 41.88,
 'longitude': -87.63,
 'start_date': '2025-01-01',
 'end_date': '2025-06-30',
 'timezone': 'America/Chicago',
 'daily': 'weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours'}

In [121]:
# call Open-Meteo API
response = requests.get(BASE_URL, params=params, timeout=60)
response.raise_for_status()

data = response.json()

print("Status code:", response.status_code)
print("Top-level keys:", list(data.keys()))

Status code: 200
Top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']


In [122]:
# save raw JSON
with open(RAW_DIR / "openmeteo_daily_raw.json", "w") as f:
    json.dump(data, f, indent=2)

print("Saved raw JSON to:", RAW_DIR / "openmeteo_daily_raw.json")

Saved raw JSON to: /Users/aanchal/data_raw/openmeteo_daily_raw.json


In [123]:
# convert daily weather block to dataframe
daily_weather = pd.DataFrame(data["daily"])

daily_weather.head()

,time,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours
0,2025-01-01,71,-1.1,0.8,-3.0,0.1,0.0,0.07,1.0
1,2025-01-02,71,-2.4,-0.3,-4.2,0.2,0.0,0.14,2.0
2,2025-01-03,3,-4.9,-3.0,-6.9,0.0,0.0,0.00,0.0
3,2025-01-04,3,-8.0,-5.1,-10.9,0.0,0.0,0.00,0.0
4,2025-01-05,71,-6.6,-2.3,-9.0,0.4,0.1,0.21,3.0


In [124]:
# clean and prepare columns
daily_weather["time"] = pd.to_datetime(daily_weather["time"])
daily_weather = daily_weather.rename(columns={"time": "crash_day"})

# derived flags
daily_weather["wet_day"] = (daily_weather["precipitation_sum"] > 0).astype(int)
daily_weather["snow_day"] = (daily_weather["snowfall_sum"] > 0).astype(int)

daily_weather.head()

,crash_day,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,wet_day,snow_day
0,2025-01-01,71,-1.1,0.8,-3.0,0.1,0.0,0.07,1.0,1,1
1,2025-01-02,71,-2.4,-0.3,-4.2,0.2,0.0,0.14,2.0,1,1
2,2025-01-03,3,-4.9,-3.0,-6.9,0.0,0.0,0.00,0.0,0,0
3,2025-01-04,3,-8.0,-5.1,-10.9,0.0,0.0,0.00,0.0,0,0
4,2025-01-05,71,-6.6,-2.3,-9.0,0.4,0.1,0.21,3.0,1,1


In [125]:
# quick structure check
print("Shape:", daily_weather.shape)
print("\nColumns:")
print(daily_weather.columns.tolist())

daily_weather.describe(include="all")

Shape: (181, 11)

Columns:
['crash_day', 'weather_code', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'precipitation_hours', 'wet_day', 'snow_day']


,crash_day,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,wet_day,snow_day
count,181,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000
mean,2025-04-01 00:00:00,33.325967,7.114917,11.412707,3.193923,2.420442,2.156906,0.184475,3.077348,0.519337,0.165746
min,2025-01-01 00:00:00,0.000000,-18.400000,-15.400000,-21.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025-02-15 00:00:00,3.000000,-1.100000,2.700000,-3.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2025-04-01 00:00:00,51.000000,6.600000,11.400000,3.800000,0.100000,0.000000,0.000000,1.000000,1.000000,0.000000
75%,2025-05-16 00:00:00,63.000000,15.000000,20.300000,9.900000,1.500000,0.900000,0.000000,5.000000,1.000000,0.000000
max,2025-06-30 00:00:00,75.000000,30.900000,35.600000,25.900000,41.100000,41.100000,6.160000,20.000000,1.000000,1.000000
std,NaN,30.138547,10.760770,11.523474,10.289505,5.761083,5.698501,0.736238,4.545399,0.501012,0.372884


In [126]:
# save processed weather csv
daily_weather.to_csv(PROCESSED_DIR / "openmeteo_daily_jan_jun_2025.csv", index=False)

print("Saved processed weather CSV to:", PROCESSED_DIR / "openmeteo_daily_jan_jun_2025.csv")

Saved processed weather CSV to: /Users/aanchal/data_processed/openmeteo_daily_jan_jun_2025.csv


## Open-Meteo Quality Assessment

The Open-Meteo dataset is assessed across the same quality dimensions applied to the crash data, adapted to the characteristics of an API-based meteorological archive:

- **Completeness:** Date coverage across the full project window (181 days); missing values in precipitation and weather code fields.
- **Validity:** Plausibility checks on precipitation and temperature values against expected Chicago climate ranges for January–June.
- **Consistency:** Internal consistency between weather codes and precipitation/snowfall flags (e.g. snow codes should align with snowfall_sum > 0).
- **Credibility:** Open-Meteo historical data are derived from ERA5 reanalysis, a well-documented meteorological model maintained by the European Centre for Medium-Range Weather Forecasts (ECMWF). Data are provided under CC BY 4.0.
- **Timeliness:** Data were retrieved on March 25, 2026, covering a window ending June 30, 2025 — a lag of approximately 9 months. For a frozen historical snapshot used in a retrospective quality assessment, this lag is acceptable and does not affect the validity of the data for this use case.

In [128]:
# quick report-ready summary
weather_summary = pd.DataFrame({
    "metric": [
        "row_count",
        "column_count",
        "min_day",
        "max_day",
        "wet_day_count",
        "snow_day_count",
        "missing_precipitation_sum",
        "missing_weather_code"
    ],
    "value": [
        daily_weather.shape[0],
        daily_weather.shape[1],
        daily_weather["crash_day"].min(),
        daily_weather["crash_day"].max(),
        int(daily_weather["wet_day"].sum()),
        int(daily_weather["snow_day"].sum()),
        int(daily_weather["precipitation_sum"].isna().sum()),
        int(daily_weather["weather_code"].isna().sum())
    ]
})

weather_summary

,metric,value
0,row_count,181
1,column_count,11
2,min_day,2025-01-01 00:00:00
3,max_day,2025-06-30 00:00:00
4,wet_day_count,94
5,snow_day_count,30
6,missing_precipitation_sum,0
7,missing_weather_code,0


In [129]:
weather_summary.to_csv(PROCESSED_DIR / "openmeteo_weather_summary.csv", index=False)
print("Saved weather summary to:", PROCESSED_DIR / "openmeteo_weather_summary.csv")

Saved weather summary to: /Users/aanchal/data_processed/openmeteo_weather_summary.csv


In [130]:
openmeteo_profile = pd.DataFrame({
    "metric": [
        "row_count",
        "min_day",
        "max_day",
        "missing_precipitation_sum",
        "missing_weather_code",
        "wet_day_count",
        "dry_day_count",
        "snow_day_count",
        "mean_precipitation_sum",
        "max_precipitation_sum",
        "mean_temperature_2m_mean",
        "min_temperature_2m_min",
        "max_temperature_2m_max"
    ],
    "value": [
        daily_weather.shape[0],
        daily_weather["crash_day"].min(),
        daily_weather["crash_day"].max(),
        int(daily_weather["precipitation_sum"].isna().sum()),
        int(daily_weather["weather_code"].isna().sum()),
        int((daily_weather["wet_day"] == 1).sum()),
        int((daily_weather["wet_day"] == 0).sum()),
        int((daily_weather["snow_day"] == 1).sum()),
        float(daily_weather["precipitation_sum"].mean()),
        float(daily_weather["precipitation_sum"].max()),
        float(daily_weather["temperature_2m_mean"].mean()),
        float(daily_weather["temperature_2m_min"].min()),
        float(daily_weather["temperature_2m_max"].max())
    ]
})

openmeteo_profile

,metric,value
0,row_count,181
1,min_day,2025-01-01 00:00:00
2,max_day,2025-06-30 00:00:00
3,missing_precipitation_sum,0
4,missing_weather_code,0
5,wet_day_count,94
6,dry_day_count,87
7,snow_day_count,30
8,mean_precipitation_sum,2.420442
9,max_precipitation_sum,41.1


In [131]:
openmeteo_profile.to_csv(PROCESSED_DIR / "openmeteo_profile_metrics.csv", index=False)
print(PROCESSED_DIR / "openmeteo_profile_metrics.csv")

/Users/aanchal/data_processed/openmeteo_profile_metrics.csv


In [132]:
openmeteo_numeric_summary = daily_weather[
    ["precipitation_sum", "rain_sum", "snowfall_sum", "precipitation_hours",
     "temperature_2m_mean", "temperature_2m_min", "temperature_2m_max"]
].describe().T

openmeteo_numeric_summary

,count,mean,std,min,25%,50%,75%,max
precipitation_sum,181.0,2.420442,5.761083,0.0,0.0,0.1,1.5,41.10
rain_sum,181.0,2.156906,5.698501,0.0,0.0,0.0,0.9,41.10
snowfall_sum,181.0,0.184475,0.736238,0.0,0.0,0.0,0.0,6.16
precipitation_hours,181.0,3.077348,4.545399,0.0,0.0,1.0,5.0,20.00
temperature_2m_mean,181.0,7.114917,10.760770,-18.4,-1.1,6.6,15.0,30.90
temperature_2m_min,181.0,3.193923,10.289505,-21.9,-3.5,3.8,9.9,25.90
temperature_2m_max,181.0,11.412707,11.523474,-15.4,2.7,11.4,20.3,35.60


In [133]:
openmeteo_numeric_summary.to_csv(PROCESSED_DIR / "openmeteo_numeric_summary.csv")
print(PROCESSED_DIR / "openmeteo_numeric_summary.csv")

/Users/aanchal/data_processed/openmeteo_numeric_summary.csv


In [134]:
# ── PRECIPITATION PLAUSIBILITY CHECK ────────────────────────────────
precip_flags = daily_weather[daily_weather["precipitation_sum"] > 100]
print(f"Days with precipitation > 100mm (implausible): {len(precip_flags)}")

temp_low_flags = daily_weather[daily_weather["temperature_2m_min"] < -30]
temp_high_flags = daily_weather[daily_weather["temperature_2m_max"] > 40]
print(f"Days with min temp < -30C (implausible for Chicago): {len(temp_low_flags)}")
print(f"Days with max temp > 40C (implausible for Chicago): {len(temp_high_flags)}")

plausibility_summary = pd.DataFrame({
    "check": ["precip > 100mm", "temp_min < -30C", "temp_max > 40C"],
    "flagged_days": [len(precip_flags), len(temp_low_flags), len(temp_high_flags)]
})
plausibility_summary.to_csv(PROCESSED_DIR / "openmeteo_plausibility_checks.csv", index=False)
print(plausibility_summary)

Days with precipitation > 100mm (implausible): 0
Days with min temp < -30C (implausible for Chicago): 0
Days with max temp > 40C (implausible for Chicago): 0
             check  flagged_days
0   precip > 100mm             0
1  temp_min < -30C             0
2   temp_max > 40C             0


In [135]:
# ── WEATHER CODE DISTRIBUTION ────────────────────────────────────────
# WMO weather code groupings (simplified)
wmo_map = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Icy fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
    77: "Snow grains", 80: "Slight showers", 81: "Moderate showers",
    82: "Violent showers", 85: "Slight snow showers", 86: "Heavy snow showers",
    95: "Thunderstorm"
}

weather_code_counts = daily_weather["weather_code"].value_counts().reset_index()
weather_code_counts.columns = ["weather_code", "count"]
weather_code_counts["description"] = weather_code_counts["weather_code"].map(wmo_map).fillna("Other")
print("Weather code distribution:")
print(weather_code_counts)

# Consistency check: snow codes vs snowfall_sum > 0
snow_codes = [71, 73, 75, 77, 85, 86]
snow_code_days = daily_weather["weather_code"].isin(snow_codes).sum()
snowfall_days = (daily_weather["snowfall_sum"] > 0).sum()
print(f"\nDays with snow weather code: {snow_code_days}")
print(f"Days with snowfall_sum > 0: {snowfall_days}")

weather_code_counts.to_csv(PROCESSED_DIR / "openmeteo_weather_code_distribution.csv", index=False)

Weather code distribution:
    weather_code  count       description
0              3     76          Overcast
1             51     21     Light drizzle
2             71     19       Slight snow
3             63     15     Moderate rain
4             53     11  Moderate drizzle
5             61     10       Slight rain
6             73      7     Moderate snow
7             65      4        Heavy rain
8             75      4        Heavy snow
9              0      4         Clear sky
10             1      4      Mainly clear
11             2      3     Partly cloudy
12            55      3     Dense drizzle

Days with snow weather code: 30
Days with snowfall_sum > 0: 30


In [136]:
# ── MONTHLY VARIABILITY ──────────────────────────────────────────────
daily_weather["month"] = daily_weather["crash_day"].dt.month

monthly = daily_weather.groupby("month").agg(
    avg_precip=("precipitation_sum", "mean"),
    total_precip=("precipitation_sum", "sum"),
    avg_temp=("temperature_2m_mean", "mean"),
    wet_days=("wet_day", "sum"),
    snow_days=("snow_day", "sum")
).reset_index()

month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun"}
monthly["month_name"] = monthly["month"].map(month_names)
print("Monthly weather summary:")
print(monthly.to_string(index=False))
monthly.to_csv(PROCESSED_DIR / "openmeteo_monthly_summary.csv", index=False)

Monthly weather summary:
 month  avg_precip  total_precip  avg_temp  wet_days  snow_days month_name
     1    1.945161          60.3 -5.390323        13         11        Jan
     2    0.846429          23.7 -3.246429        14         10        Feb
     3    3.035484          94.1  6.051613        18          8        Mar
     4    2.156667          64.7  9.706667        18          1        Apr
     5    2.300000          71.3 13.035484        16          0        May
     6    4.133333         124.0 22.096667        15          0        Jun


## Open-Meteo Validity Summary

All plausibility checks passed:
- Zero days with precipitation > 100mm (implausible daily total for Chicago)
- Zero days with minimum temperature < -30°C
- Zero days with maximum temperature > 40°C
- Monthly temperature progression (-5.4°C mean in January to 22.1°C in June) is consistent with expected Chicago climate patterns
- Snow days concentrated in January–March (29 of 30 total snow days), zero in May–June — consistent with seasonal expectations
- Perfect internal consistency between snow weather codes and snowfall_sum > 0 (both = 30 days)

The Open-Meteo dataset passes all validity checks and is assessed as complete and credible for use as an external weather reference in this project.